In [ ]:
import os
import pandas as pd
import re
from dotenv import load_dotenv
from sqlalchemy import create_engine
import pickle # lưu model sau khi training

# Import tf-idf và cosine similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Import success!")

Tất cả công cụ đã được import!


In [ ]:
def load_data():
    """Kết nối DB"""
    load_dotenv()
    
    try:
        connection_string = f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_DATABASE')}"
        engine = create_engine(connection_string)
        print("Kết nối CSDL PostgreSQL thành công!")
    except Exception as e:
        print(f"LỖI: Không thể kết nối CSDL. Lỗi: {e}")
        return None

    # Các câu lệnh SQL
    sql_movies = "SELECT id, title FROM Movies"
    sql_genres = "SELECT movie_id, STRING_AGG(G.name, ', ') AS genres FROM Movie_Genres MG JOIN Genres G ON MG.genre_id = G.id GROUP BY MG.movie_id"
    sql_actors = "SELECT movie_id, STRING_AGG(A.name, ', ') AS actors FROM Movie_Actors MA JOIN Actors A ON MA.actor_id = A.id GROUP BY MA.movie_id"
    sql_directors = "SELECT movie_id, STRING_AGG(D.name, ', ') AS directors FROM Movie_Directors MD JOIN Directors D ON MD.director_id = D.id GROUP BY MD.movie_id"

    try:
        print("Đang tải dữ liệu (movies, genres, actors, directors)...")
        df_movies = pd.read_sql(sql_movies, engine)
        df_genres = pd.read_sql(sql_genres, engine)
        df_actors = pd.read_sql(sql_actors, engine)
        df_directors = pd.read_sql(sql_directors, engine)

        # Gộp (Merge) tất cả
        df_full = pd.merge(df_movies, df_genres, left_on='id', right_on='movie_id', how='left')
        df_full = pd.merge(df_full, df_actors, left_on='id', right_on='movie_id', how='left')
        df_full = pd.merge(df_full, df_directors, left_on='id', right_on='movie_id', how='left')
        
        df_full = df_full.drop(columns=['movie_id_x', 'movie_id_y', 'movie_id'])
        print("Tải và gộp dữ liệu thành công!")
        return df_full

    except Exception as e:
        print(f"LỖI: Không thể tải dữ liệu. Lỗi: {e}")
        return None

In [16]:
def clean_text(text):
    """Hàm dọn dẹp (xóa dấu cách, viết thường)"""
    if text is None:
        return ""
    names = [re.sub(r'\s+', '', name).lower() for name in text.split(', ')]
    return " ".join(names)

In [17]:
def create_soup(data_row):
    """Hàm trộn các nguyên liệu (genres, actors, directors)"""
    genres = clean_text(data_row['genres'])
    actors = clean_text(data_row['actors'])
    directors = clean_text(data_row['directors'])
    return f"{genres} {actors} {directors}"

In [18]:
# Load data from database
data = load_data()

data.head()

Đang tải biến môi trường...
Kết nối CSDL PostgreSQL thành công!
Đang tải dữ liệu (movies, genres, actors, directors)...
Tải và gộp dữ liệu thành công!


,id,title,genres,actors,directors
0,278,The Shawshank Redemption,"Drama, Crime","Clancy Brown, Tim Robbins, Morgan Freeman, Bob...",Frank Darabont
1,1311031,Demon Slayer: Kimetsu no Yaiba Infinity Castle,"Fantasy, Animation, Action, Thriller","Natsuki Hanae, Hiro Shimono, Yoshitsugu Matsuo...",Haruo Sotozaki
2,1084242,Zootopia 2,"Adventure, Animation, Comedy, Mystery, Family","Andy Samberg, Ginnifer Goodwin, Jason Bateman,...",Jared Bush
3,507244,Afterburn,"Action, Comedy, Science Fiction","Dave Bautista, Samuel L. Jackson, Daniel Bernh...",J.J. Perry
4,1269021,Wake,"Horror, Thriller","Sadie Earegood, Susan Reno, Jake Holley, Sarab...",Thom Arizmendi


In [19]:
# Tiền xử lý (Tạo cột 'soup': nguyên liệu để vectorize)
        
# Fill các ô (NaN/NULL) bằng chuỗi rỗng ""
data['genres'] = data['genres'].fillna('')
data['actors'] = data['actors'].fillna('')
data['directors'] = data['directors'].fillna('')

# Tạo cột "soup"
# .apply() sẽ chạy hàm 'create_soup' cho từng hàng
data['soup'] = data.apply(create_soup, axis=1)

print("Đã tạo 'soup' thành công!")

# Hiển thị kết quả 
data[['id', 'title', 'soup']].head()

Đã tạo 'soup' thành công!


,id,title,soup
0,278,The Shawshank Redemption,drama crime clancybrown timrobbins morganfreem...
1,1311031,Demon Slayer: Kimetsu no Yaiba Infinity Castle,fantasy animation action thriller natsukihanae...
2,1084242,Zootopia 2,adventure animation comedy mystery family andy...
3,507244,Afterburn,action comedy sciencefiction davebautista samu...
4,1269021,Wake,horror thriller sadieearegood susanreno jakeho...


In [20]:
# Khởi tạo TF-IDF
# stop_words='english' sẽ tự động bỏ các từ vô nghĩa (the, a, in)
tfidf = TfidfVectorizer(stop_words='english')

# 2. "Dạy" (fit) và "Biến đổi" (transform) cột 'soup'
# Kết quả là một ma trận toán học khổng lồ
tfidf_matrix = tfidf.fit_transform(data['soup'])

# 3. In ra kết quả
# (ví dụ: 492 phim, 5000 "nguyên liệu" (từ) độc nhất)
print("Ma trận TF-IDF (Không gian Đặc trưng) đã được tạo:")
print(tfidf_matrix.shape)

Ma trận TF-IDF (Không gian Đặc trưng) đã được tạo:
(1554, 7165)


In [21]:
# Tính toán Độ tương đồng Cosine
# Đo độ tương đồng giữa TẤT CẢ phim với TẤT CẢ phim khác

# 1. Chạy hàm cosine_similarity trên ma trận TF-IDF
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print("Ma trận tương đồng (Cosine Similarity) đã được tạo:")
print(cosine_sim.shape)

print("Một phần của ma trận (5x5):")
print(cosine_sim[0:5, 0:5])

Ma trận tương đồng (Cosine Similarity) đã được tạo:
(1554, 1554)
Một phần của ma trận (5x5):
[[1.         0.         0.         0.         0.        ]
 [0.         1.         0.02834145 0.0143267  0.01575328]
 [0.         0.02834145 1.         0.01412706 0.        ]
 [0.         0.0143267  0.01412706 1.         0.        ]
 [0.         0.01575328 0.         0.         1.        ]]


In [22]:
indices = pd.Series(data.index, index=data['id']).drop_duplicates()
indices.head()

id
278        0
1311031    1
1084242    2
507244     3
1269021    4
dtype: int64

In [23]:
# Xây dựng Hàm Gợi ý
# Chúng ta cần một cách để tra cứu ID phim -> Vị trí (index) của nó trong ma trận
indices = pd.Series(data.index, index=data['id']).drop_duplicates()

def get_recommendations(movie_id, cosine_sim_matrix=cosine_sim, data=data):
    """
    Hàm nhận vào ID phim và trả về 10 phim giống nhất
    """
    try:
        # 1. Lấy vị trí (index) của phim trong ma trận
        idx = indices[movie_id]

        # 2. Lấy danh sách điểm tương đồng của phim đó với TẤT CẢ phim khác
        # enumerate() sẽ tạo thành cặp (index, score)
        sim_scores = list(enumerate(cosine_sim_matrix[idx]))

        # 3. Sắp xếp danh sách (lấy điểm cao nhất)
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

        # 4. Lấy 15 phim đầu tiên (bỏ qua phim đầu tiên, vì đó là chính nó)
        sim_scores = sim_scores[1:16]

        # 5. Lấy ID của 15 phim đó
        movie_indices = [i[0] for i in sim_scores]
        
        # 6. Trả về tên của 10 phim
        return data['title'].iloc[movie_indices]

    except KeyError:
        return f"Lỗi: Không tìm thấy ID phim {movie_id} trong dữ liệu."
    except Exception as e:
        return f"Lỗi: {e}"

print("Hàm 'get_recommendations' đã sẵn sàng.")

Hàm 'get_recommendations' đã sẵn sàng.


In [24]:
# Ô 7: Chạy thử gợi ý!

# ví dụ phim The Drark Knight là 155
test_movie_id = 155 

print(f"--- Gợi ý cho phim (ID: {test_movie_id}) ---")
recommendations = get_recommendations(test_movie_id)
print(recommendations)



--- Gợi ý cho phim (ID: 155) ---
1158                 Batman Begins
1278                  The Prestige
743          The Dark Knight Rises
253                   Interstellar
600         Muzzle: City of Wolves
1416               American Psycho
462                 Public Enemies
1042    10 Things I Hate About You
29                     Oppenheimer
603           Billion Dollar Brain
964                      Inception
1535                       Memento
1012                  Curtain Call
651                      Amsterdam
147         Thor: Love and Thunder
Name: title, dtype: object


In [25]:
# Ô 8: Lưu mô hình (bộ não) ra file

import pickle

# 1. Lưu ma trận tương đồng (cosine_sim)
#    Đây là "bộ não" toán học
pickle.dump(cosine_sim, open('cosine_similarity_matrix.pkl', 'wb'))

# 2. Lưu bảng dữ liệu (data)
#    Chúng ta cần bảng này để tra cứu 'title' từ 'id'
#    Chúng ta chỉ lưu các cột cần thiết
data_to_save = data[['id', 'title']]
pickle.dump(data_to_save, open('movie_data.pkl', 'wb'))

print("Đã lưu thành công 2 file: 'cosine_similarity_matrix.pkl' và 'movie_data.pkl'")

Đã lưu thành công 2 file: 'cosine_similarity_matrix.pkl' và 'movie_data.pkl'
